# L33 · 正弦波生成 — `x[n]=A·sin(2πfn/sr)`，亲手实现并对齐 aurora.audio.sine

**目标**：实现 `x[n] = A·sin(2π·f·n/sr)`，手动写出每一步，再用 `np.allclose` 验证与 `aurora.audio.sine` 逐点一致。

`aurora.audio.sine` 是后续 STFT 和 mel 特征提取的测试信号来源，`my_sine` 实现的正是同一套公式。

## 本课剧情：追踪圆周上的影子

`n` 是采样点编号，`2π·f·n/sr` 把编号换算成弧度角，`sin` 把角度投影到 [-1, 1]，输出是 float64 数组。

实验从 `sample_rate=8` 的短序列开始，让 `n`、`angle`、`wave` 的数值都可以直接打印出来。

## 1. 导入工具 + 仓库的"标准答案"

这一段是为了让你一边写自己的版本，一边随时拿仓库版本对照。学习时最怕“只看答案，不知道答案为什么长这样”。所以我们会先把参考实现摆在旁边，再逐步拆公式。


## 实验入口：把声音拆成可观察的数组

用 `sample_rate=8`、`duration=1.0`、`freq=2.0` 生成 8 个采样点，每一步的中间量都打印出来，可以直接看到 `n`、`t`、`angle`、`wave` 的数值对应关系。

In [ ]:
import numpy as np
from aurora.audio import sine as reference_sine  # 仓库实现 = 你的对照答案
print('就绪')

## 动手观察：序列怎样一步步变成信号

观察 `n`、`t`、`angle` 三个中间数组：`angle` 是 `t` 乘以 `2π·freq`，当 `freq=2` 时，1 秒内 `angle` 从 0 走到 4π，对应两个完整周期。

In [ ]:
import numpy as np

sample_rate = 8
duration = 1.0
freq = 2.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
wave = np.sin(angle)

print('N =', N)
print('采样点编号 n =', n)
print('时间轴 t =', np.round(t, 3))
print('角度 angle =', np.round(angle, 3))
print('sin(angle) =', np.round(wave, 3))


## 代码实验：遍历频率、振幅和相位

`freq` 加倍时，相同时间段内的过零点数量也加倍；`amplitude` 只改变数值范围，不影响波形形状。

In [ ]:
import numpy as np

sample_rate = 16
duration = 0.5
t = np.arange(round(duration * sample_rate)) / sample_rate

for freq in [1, 2, 4]:
    wave = np.sin(2 * np.pi * freq * t)
    print(f'freq={freq}Hz ->', np.round(wave, 3))

for amplitude in [0.5, 1.0, 2.0]:
    wave = amplitude * np.sin(2 * np.pi * 2 * t)
    print(f'amplitude={amplitude} -> min={wave.min():.1f}, max={wave.max():.1f}')


## 2. 公式拆解

`x[n] = amplitude · sin(2π · freq · n / sample_rate)`

先别急着背，把它拆成三层看：

1. **先有采样点编号**：`n = 0, 1, 2, ...`
2. **再把编号变成角度**：`angle = 2π · freq · n / sample_rate`
3. **最后把角度喂给 sin**：`sin(angle)` 给出每个点的波形值

- `n` 是第几个采样点：`0, 1, 2, ...`
- `2π · freq · n / sr` 是这个点对应的**角度（弧度）**
- `sin(角度)` 给出该点的波形值

可以把它想成"排队"这件事：`n` 是队号，`angle` 是每个人站到圆周上的位置，`sin` 是把圆上的位置投影到上下方向，于是就得到波形。

`n / sample_rate` 在数值上等价于时间 `t`，所以公式也可以写成 `A·sin(2π·f·t)`。但代码里用整数编号 `n` 除以 `sample_rate`，而不是用 `np.linspace(0, duration, N)` 生成浮点时间轴：`np.linspace` 在端点附近存在舍入误差，而整数序列的一次除法更干净。这正是仓库 `aurora.audio.sine`（`src/aurora/audio/core.py`）的选择，也是 `my_sine` 要用同样方式才能让 `np.allclose` 误差落在 1e-15 量级的原因。


## 3. ✏️ 你的任务：实现 `my_sine`

这一题最关键的不是公式本身，而是把公式翻译成代码时，知道每个变量在代码里扮演什么角色：

- `freq`：频率，决定波动快慢
- `duration`：时长，决定序列有多长
- `sample_rate`：采样率，决定每秒取多少点
- `n` 或 `t`：位置序列，代表每个采样点的编号或时间
- `amplitude`：振幅，决定波形上下摆动的幅度

**推理路线**：
1. 先算出总点数：`N = round(duration * sample_rate)`，这是返回数组的长度。
2. 生成整数编号序列：`n = np.arange(N)`，从 0 到 N-1，不含端点 N。
3. 把编号换算成角度：`angle = 2*np.pi*freq*n/sample_rate`，其中 `n/sample_rate` 就是每个点的时间 `t`。
4. 返回 `amplitude * np.sin(angle)`——振幅只是一个缩放因子，不影响波形形状。

**参考输入输出**：`freq=1, duration=1, sample_rate=8` → 8 个点，`angle=[0, π/4, π/2, 3π/4, π, 5π/4, 3π/2, 7π/4]`，`sin≈[0, 0.707, 1, 0.707, 0, -0.707, -1, -0.707]`

<details><summary>点击查看参考实现</summary>

```python
def my_sine(freq, duration, sample_rate, amplitude=1.0):
    N = round(duration * sample_rate)
    n = np.arange(N)
    angle = 2 * np.pi * freq * n / sample_rate
    return amplitude * np.sin(angle)
```

</details>


### 写 `my_sine` 前明确三件事

- 输入：`freq`（Hz）、`duration`（秒）、`sample_rate`（点/秒）、`amplitude`（默认 1.0）
- 关键步骤：`N = round(duration * sample_rate)`，`n = np.arange(N)`，`angle = 2*np.pi*freq*n/sample_rate`
- 返回：长度为 `N` 的 float64 数组，值域 `[-amplitude, amplitude]`

In [ ]:
def my_sine(freq, duration, sample_rate, amplitude=1.0):
    # ✏️ TODO: 按公式实现并返回正弦波
    return None  # ← 改这里


## 4. 对答案：和仓库实现逐点比较

这里要看的不只是“有没有报错”，还要看三个层面的对应关系：

- **长度是否一致**：说明你有没有生成正确数量的采样点
- **每个点是否一致**：说明你有没有把公式写对
- **误差是否极小**：说明数值计算里的浮点误差是否在正常范围

如果这里没过，不要急着改 `sin` 本身，先检查 `n` 是不是从 0 开始、`N` 算得对不对、采样率是不是除错了。


In [ ]:
freq, dur, sr = 440.0, 1.0, 16000
mine = my_sine(freq, dur, sr)
ref = reference_sine(freq, duration=dur, sample_rate=sr)
max_diff = float(np.max(np.abs(mine - ref)))
print('我的 shape:', mine.shape, '| 标准 shape:', ref.shape)
print('最大逐点误差:', f'{max_diff:.2e}')
assert mine.shape == ref.shape, '长度应一致'
assert max_diff < 1e-6, '数值应几乎一致'
print('\n✅ 对答案通过：你的正弦波和仓库一致。')

## 5. 画出来看看（前 50 个采样点）

图像能直接显示序列的时间变化。你看到的不是“数组”，而是“这个波在时间上怎么上下摆动”。

前 50 个采样点只是一个很小的窗口，但它能帮你看出：
- 波形是不是连续的
- 振幅是不是对的
- 频率是不是符合预期

如果你愿意，还可以把 `mine[:50]` 和 `ref[:50]` 叠在一起看，这样更容易发现偏差。


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 3))
plt.plot(mine[:50], marker='o', ms=3)
plt.title('440 Hz sine — first 50 samples')
plt.xlabel('sample n'); plt.ylabel('amplitude')
plt.tight_layout(); plt.show()

**🎉 完成后**：`git commit -m 'learn: Day 2 my_sine'`

## 🎨 图示：440Hz 正弦的前 50 个采样点

这一段不是“装饰”，而是 notebook 最强的地方：**把公式变成图，把图再变成直觉**。当你以后写别的波形、别的序列、别的信号处理代码时，这种“先生成、再比较、再可视化”的习惯会一直有用。


In [ ]:
from aurora.audio import sine
import aviz; aviz.style()
aviz.waveform(sine(440.0, duration=50/16000, sample_rate=16000), stem=True,
              title='440 Hz 正弦 — 前 50 个采样点');

In [ ]:
sr = 32
duration = 1.0
t = np.arange(round(duration * sr)) / sr

for freq in [1, 2, 4, 8]:
    x = np.sin(2*np.pi*freq*t)
    zero_crossings = np.sum(np.diff(np.signbit(x)) != 0)
    print(f'freq={freq:>2}Hz | 前8点={np.round(x[:8], 2)} | 过零次数≈{zero_crossings}')


## 参数实验：Nyquist 频率的采样极限

把 `freq` 改成 `sample_rate/2`（Nyquist 频率），观察输出交替为 `[0, 1, 0, -1, ...]` 或 `[0, 0, 0, ...]`（取决于相位）——每个完整周期只剩两个采样点，正好够表示，但已是极限。这是 Day 3（混叠）的预习：当 `freq > sample_rate/2` 时，采样点不够，高频成分会被折叠成低频噪声。

In [ ]:
sr = 32
duration = 1.0
t = np.arange(round(duration * sr)) / sr

for freq in [1, 2, 4, 8]:
    x = np.sin(2*np.pi*freq*t)
    print(f'freq={freq:>2}Hz | 前8点={np.round(x[:8], 2)} | max={x.max():.1f}')


## 本课收束

现在可以用 `my_sine(freq, duration, sample_rate)` 生成任意频率的正弦波。

`np.allclose(my_sine(...), reference_sine(...))` 返回 `True`，说明两个实现用的是同一套公式。

`aurora.audio.sine` 在 STFT 和 mel 特征提取的测试中生成标准信号，`my_sine` 是它的手动复现。

下一节（Day 3）会对这条正弦波降低采样率，观察高于 Nyquist 的频率如何被折叠到低频。

In [ ]:
# 小检查：从短序列开始，确认每一步输出
import numpy as np

sample_rate = 8
duration = 1.0
freq = 1.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
x = np.sin(angle)

print('1) N 应该是多少？', N)
print('2) n 是采样点编号：', n)
print('3) t 是每个点的秒数：', np.round(t, 3))
print('4) angle 是每个点在圆上的角度：', np.round(angle, 3))
print('5) x 是最终波形：', np.round(x, 3))
